# POMCA — Plan de Ordenación y Manejo de Cuencas Hidrográficas

> **Contexto de dominio:** [`docs/fuentes/pomca.md`](../../docs/fuentes/pomca.md)
> **Bloque:** A — Gestión | **Línea:** `pomca`
> **Variable principal:** `caudal` (m³/s) — Oferta hídrica superficial de la cuenca
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST
> **Flujo:** `Plan.md` sección 3 — ciclo estadístico completo.

---

## ¿Qué es el POMCA y por qué es el instrumento rector en Colombia?

El **Plan de Ordenación y Manejo de Cuencas Hidrográficas (POMCA)** es el instrumento rector de planificación ambiental territorial en Colombia. Está diseñado para articular el uso coordinado y sostenible del suelo, el agua, la flora y la fauna dentro de una cuenca hidrográfica. Jurídicamente, es una **norma de superior jerarquía**: los Planes de Ordenamiento Territorial (POT) municipales y departamentales deben subordinarse a sus determinantes ambientales.

El POMCA se desarrolla en un ciclo de seis fases: aprestamiento, diagnóstico, prospectiva y zonificación ambiental, formulación, ejecución, y seguimiento y evaluación. El análisis estadístico del caudal es la columna vertebral del diagnóstico: permite establecer el balance oferta-demanda, identificar años hidrológicos críticos y estimar la presión sobre el recurso a futuro.

## Instituciones responsables

| Institución | Rol |
|---|---|
| **MADS** | Dicta política y marcos regulatorios nacionales |
| **CARs** | Responsables técnicas y administrativas de elaborar y ejecutar el POMCA |
| **Consejo de Cuenca** | Instancia participativa: alcaldías, comunidades, gremios, universidades |
| **IDEAM / IGAC / SGC** | Proveen cartografía, climatología y reportes de calidad ambiental |

## Indicadores hídricos que se calculan en el POMCA

| Indicador | Descripción | Umbral de alerta |
|---|---|---|
| **IUA** (Índice de Uso del Agua) | Demanda / Oferta disponible × 100 | >20% moderado, >50% alto, >100% crítico |
| **IRH** (Ret. y Regulación Hídrica) | Capacidad de la cuenca para mantener caudal base | < 0.5 = regulación muy baja |
| **IVH** (Vulnerabilidad por Desabastecimiento) | Combina IUA + IRH | Alto IUA + Bajo IRH = riesgo crítico |
| **IACAL** (Alteración Potencial Calidad) | Presión de vertimientos vs. oferta | Valores altos = riesgo contaminación |
| **ICA** (Calidad del Agua) | Estado fisicoquímico ponderado | 0-1; < 0.5 = calidad inaceptable |

## Preguntas que responde este análisis

1. ¿Cuál es el régimen hidrológico de la cuenca? ¿Predomina un período bimodal, unimodal o equinoccial?
2. ¿El IUA mensual supera el umbral crítico del 100%? ¿En qué meses?
3. ¿Hay una tendencia significativa de cambio en el caudal medio anual (Mann-Kendall)?
4. ¿Cómo influye ENSO en la variabilidad interanual del caudal de la cuenca?
5. ¿Qué modelo estadístico proyecta mejor el caudal para escenarios de planificación a 10 años?

---

**Antes de ejecutar:** Leer `docs/fuentes/pomca.md` para comprender los indicadores IUA, IRH e IVH, la normativa del Decreto 1076/2015 y las fuentes de datos DHIME/SIRH del IDEAM.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "pomca"
VARIABLE = "caudal"
UNIDAD = "m³/s"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio

Esta celda carga la ficha técnica de la línea `pomca` directamente desde `docs/fuentes/`. La ficha contiene el resumen ejecutivo del POMCA como instrumento rector, las variables ambientales de la cuenca, los indicadores hídricos y la normativa aplicable.

Se recomienda revisar especialmente:
- La tabla de **Variables ambientales**: precipitación, temperatura, caudal medio diario (IDEAM-DHIME), coberturas (Corine Land Cover), calidad del agua.
- Los **indicadores**: IUA (demanda/oferta), IRH (regulación), IVH (vulnerabilidad desabastecimiento), IACAL (presión contaminación), ICA, IARC (huella hídrica azul).
- Las **fuentes de datos**: series hidrológicas y climatológicas del IDEAM (DHIME y SIRH), IGAC, SIAC.

> **Nota sobre outliers (ADR-002):** Los caudales extremos durante eventos de La Niña o El Niño son señal crítica para el análisis POMCA. Representan escenarios de riesgo real para el dimensionamiento de la oferta hídrica disponible. No aplicar clipping sin justificación técnica documentada.

> **Nota sobre series con brechas:** El POMCA frecuentemente trabaja con series con "baches" temporales por falta de continuidad en las estaciones hidrológicas. El módulo `preprocessing/imputation.py` con `method="linear"` o `method="seasonal"` es el primer paso recomendado antes del análisis de tendencias.

In [ ]:
ficha = DOCS_FUENTES / "pomca.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### ¿Qué datos necesito?

Para el análisis hidrológico del POMCA se requiere una serie temporal del régimen de caudales con las siguientes columnas:

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | datetime | Fecha de la medición (preferiblemente diaria, aceptable mensual) |
| `caudal` | float | Caudal medio (m³/s) — oferta hídrica superficial |
| `estacion` | str (opcional) | Código IDEAM de la estación hidrológica |
| `precipitacion` | float (opcional) | Precipitación acumulada mensual (mm) |
| `demanda` | float (opcional) | Demanda hídrica multisectorial (m³/s o m³/año) — para calcular IUA |

### Fuentes oficiales de datos en Colombia

- **IDEAM — DHIME:** [dhime.ideam.gov.co](http://dhime.ideam.gov.co) — Principal fuente de series hidrológicas diarias y mensuales. Acceso con cuenta institucional.
- **IDEAM — SIRH:** [sirh.ideam.gov.co](http://sirh.ideam.gov.co) — Concesiones de agua, demanda hídrica por cuenca.
- **Estudio Nacional del Agua (ENA):** Publicación periódica del IDEAM con IUA, IRH e IVH por cuenca y subzona hidrográfica.
- **CARs regionales:** Redes propias de aforo en tributarios menores.

### Cálculo del IUA mensual

El IUA (Índice de Uso del Agua) se calcula como:

```
IUA (%) = (Demanda_mensual / Oferta_hídrica_disponible) × 100
```

Donde la oferta disponible = oferta total - caudal ambiental (aprox. 25% del caudal medio).

| IUA (%) | Categoría | Acción recomendada |
|---|---|---|
| 0 – 20 | Bajo | Sin restricción de uso |
| 20 – 50 | Moderado | Monitoreo continuo |
| 50 – 100 | Alto | Restricción de concesiones nuevas |
| > 100 | Muy alto / Crítico | Emergencia: sobreexplotación confirmada |

### Frecuencia y horizonte recomendados

- **Frecuencia:** Diaria para modelos hidrológicos. Mensual para análisis de tendencias y ENSO.
- **Horizonte mínimo:** 20+ años para capturar ciclos ENSO completos.

> **Nota:** El dataset sintético de ejemplo usa distribución Gamma para simular caudales positivos con estacionalidad. En cuencas andinas bimodales, los picos de caudal ocurren en marzo-abril y octubre-noviembre.

In [ ]:
# df = load_csv("data/raw/pomca.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "caudal": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### ¿Qué hace `validate()` en esta línea?

La función `validate()` aplica rangos físicos específicos para la línea `pomca` definidos en `config.py`. Verifica que:

- El caudal no sea negativo (físicamente imposible).
- Los valores de caudal estén dentro del orden de magnitud esperado para el tamaño de la cuenca.
- No haya fechas duplicadas, saltos de más de un mes ni periodos vacíos superiores al umbral configurado.

### Señales de alerta en la validación

Si `val.summary()` reporta problemas:

- **Brechas de más de 3 meses:** Frecuentes en cuencas con estaciones del IDEAM discontinuadas. El POMCA requiere justificar metodológicamente la reconstrucción de datos.
- **Caudales cero en meses de lluvia:** Posibles fallos de sensor o errores de digitación. Verificar contra pluviometría de la misma época.
- **Saltos bruscos de caudal (>10× en un mes):** Pueden ser eventos extremos reales de La Niña. Conservar y documentar.

> **ADR-006:** El parámetro `linea_tematica="pomca"` activa validaciones específicas de cuencas hidrográficas, con umbrales de caudal ajustados según la superficie de drenaje y la región hidrológica.

> **ADR-002:** En cuencas andinas y de llanura inundable, los caudales máximos durante La Niña pueden superar en 5 a 10 veces el promedio histórico. Estos valores son fundamentales para el diseño de medidas de gestión del riesgo dentro del POMCA. No eliminarlos.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_pomca.html",
       title="EDA — POMCA", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### ¿Qué buscar en las gráficas?

Las visualizaciones de esta sección permiten leer el régimen hidrológico de la cuenca antes del análisis formal. En el POMCA, la visualización del caudal es el punto de partida para el diagnóstico de la oferta hídrica. Se deben identificar:

**En la serie temporal (plot_series):**
- Régimen hidrológico: ¿La cuenca tiene régimen bimodal (dos períodos húmedos), unimodal (un período húmedo) o equinoccial? En Colombia, la Región Andina es mayoritariamente bimodal; la Amazonia y el Pacífico son monomodales con lluvias casi todo el año.
- Variabilidad interanual: los años con La Niña (ONI < -0.5) deberían mostrar caudales marcadamente superiores al promedio; los años de El Niño (ONI > 0.5) deberían mostrar caudales por debajo. Esta diferenciación es clave para el diseño de los escenarios del POMCA.
- Tendencias: ¿hay una disminución del caudal base en las últimas décadas? Posible señal de pérdida de cobertura boscosa en zonas de recarga o efecto del cambio climático sobre la precipitación regional.

**En las medias estacionales (plot_seasonal_means):**
- Identificar los meses de estiaje: enero-febrero y julio-agosto en la mayoría de la Región Andina. Estos meses determinan el denominador del IUA en su versión más conservadora.
- Comparar la media con el percentil 25 (Q25): la diferencia entre el caudal medio y el Q25 da una idea de la variabilidad y el riesgo de desabastecimiento en años secos.
- Meses con IUA > 50%: si la demanda estimada de la cuenca supera la mitad del caudal disponible en meses secos, hay señal de presión alta sobre el recurso.

In [ ]:
plot_series(df, "fecha", "caudal", title="POMCA — caudal (m³/s)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "caudal", period="month")
plt.show()

## 3b. Covariable ENSO (ONI) — Por qué y con qué lag en el POMCA

### ¿Por qué el ENSO es central en el diagnóstico del POMCA?

El POMCA planifica el uso del recurso hídrico para un horizonte de 10 años. En ese horizonte, es prácticamente seguro que ocurrirán uno o más eventos de El Niño y de La Niña. Si el plan ignora la variabilidad climática interanual asociada al ENSO, los escenarios de oferta hídrica disponible serán incorrectos y las decisiones de concesión podrán ser contraproducentes.

- **El Niño:** Escenario de estrés hídrico. Reduce el caudal disponible en cuencas andinas y de la Orinoquía, incrementa el IUA automáticamente, y puede llevar a categoría "crítico" (>100%) cuencas que normalmente están en "moderado". El POMCA debe definir protocolos de contingencia para estos escenarios.
- **La Niña:** Escenario de abundancia con riesgo. Incrementa los caudales, puede recargar cuencas estresadas, pero también dispara riesgo de inundación y erosión. El POMCA debe incorporar estas amenazas en la zonificación de gestión del riesgo.

### ¿Por qué usar el lag configurado? (ADR-007)

Para la línea `pomca`, el lag ENSO se aplica sobre el caudal superficial de la cuenca. La respuesta hidrológica de una cuenca al ONI depende de su tamaño, régimen de lluvias y cobertura boscosa. El valor en `config.ENSO_LAG_MESES["pomca"]` representa el desfase promedio entre el pico del ONI y la respuesta de los caudales medios mensuales en cuencas andinas colombianas.

> **ADR-007:** `enso_lagged()` asegura que el ONI aplicado al modelo ya tiene el lag correcto. Usar ONI sin lag introduce una correlación espuria entre el estado climático actual y el caudal observado, subestimando la verdadera señal de predicción.

### Clasificación de fases ENSO (config.ENSO_THRESHOLDS)

| ONI | Fase | Impacto típico en caudales (Andes colombianos) |
|---|---|---|
| > +1.5 | El Niño fuerte | Reducción severa (-30 a -50% del promedio) |
| +0.5 a +1.5 | El Niño moderado | Reducción leve a moderada (-10 a -30%) |
| -0.5 a +0.5 | Neutro | Variabilidad normal |
| -0.5 a -1.5 | La Niña moderada | Incremento moderado (+10 a +30%) |
| < -1.5 | La Niña fuerte | Incremento severo (+30 a +70%) con riesgo de inundación |

In [ ]:
# --- Covariable ENSO (lag específico para POMCA) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="pomca")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial — Estacionariedad y tendencia en el POMCA

### ¿Por qué verificar estacionariedad antes del ARIMA? (ADR-004)

El POMCA construye proyecciones hidrológicas a 10 años. Si la serie de caudal que sirve de base tiene una tendencia no estacionaria, los modelos ARIMA mal especificados producirán pronósticos erróneos que pueden comprometer las decisiones de gestión del recurso hídrico. La ADR-004 exige el uso conjunto de ADF y KPSS.

**Regla práctica para cuencas colombianas:**
- La mayoría de las series de caudal mensual son **estacionarias en media** pero no siempre en varianza (heterocedasticidad estacional).
- Si el caudal tiene varianza que crece proporcionalmente a la media (común en cuencas de régimen tropical intenso), aplicar transformación logarítmica antes de los tests: `np.log(ts + 1)`.
- Las series con cambios estructurales (ej. construcción de un embalse, deforestación masiva en un año) pueden requerir modelos ARIMAX con variable de intervención.

### Mann-Kendall en el contexto del POMCA

La tendencia Mann-Kendall es una de las evidencias estadísticas más importantes para el POMCA:
- `trend = "decreasing"` con p < 0.05 en el caudal: el POMCA debe reconocer esta señal como posible efecto de cambio climático, pérdida de cobertura boscosa o sobreexplotación, y proponer medidas específicas de restauración.
- `slope` en m³/s/año: permite proyectar el caudal futuro bajo el escenario de tendencia continua. Si la tendencia negativa persiste, el IUA puede escalar a categoría crítica en los próximos años.
- Si la tendencia es positiva: puede indicar recuperación de cuencas por reforestación o cambio en el régimen de lluvias por cambio climático — contexto que también requiere análisis.

> **ADR-004:** ADF + KPSS son obligatorios antes de definir los parámetros d y D del modelo SARIMA para proyecciones del POMCA. Los parámetros mal seleccionados producen pronósticos que divergen a largo plazo.

In [ ]:
ts = df.set_index("fecha")["caudal"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas — IUA, IRH e IVH

### ¿Qué normas e indicadores aplican al POMCA?

El POMCA no tiene umbrales de "calidad" en el sentido fisicoquímico, sino **umbrales de gestión** que clasifican el estado del recurso hídrico según la presión de uso y la capacidad de regulación de la cuenca:

**IUA — Índice de Uso del Agua (config.IUA_THRESHOLDS):**

| IUA (%) | Categoría | Implicación para el POMCA |
|---|---|---|
| 0 – 20 | Bajo | Cuenca con disponibilidad hídrica. Sin restricciones a nuevas concesiones |
| 20 – 50 | Moderado | Cuenca bajo presión. Monitoreo intensificado. Evaluar concesiones caso a caso |
| 50 – 100 | Alto | Cuenca en estrés. Restringir nuevas concesiones. Activar medidas de ahorro |
| > 100 | Muy alto / Crítico | Sobreexplotación confirmada. Suspender concesiones. Plan de contingencia inmediato |

**IRH — Índice de Retención y Regulación Hídrica (config.IRH_THRESHOLDS):**

| IRH | Categoría |
|---|---|
| 0.0 – 0.35 | Muy baja regulación — alta variabilidad de caudales |
| 0.35 – 0.55 | Baja regulación |
| 0.55 – 0.75 | Moderada regulación |
| 0.75 – 1.00 | Alta regulación — caudal base estable |

### Cálculo del IUA mensual con los datos del notebook

Si se dispone de la columna `demanda` (m³/s) o del volumen concesionado del SIRH:
```python
from estadistica_ambiental.config import IUA_THRESHOLDS
# Caudal ambiental = 25% del caudal medio histórico (estimación conservadora)
caudal_ambiental = df["caudal"].mean() * 0.25
oferta_disponible = df["caudal"] - caudal_ambiental
iua = (df["demanda"] / oferta_disponible) * 100
categoria = pd.cut(iua, bins=[0,20,50,100,float("inf")],
                   labels=["Bajo","Moderado","Alto","Crítico"])
```

> **ADR-005:** Los umbrales IUA, IRH e IVH deben leerse siempre de `config.IUA_THRESHOLDS` y `config.IRH_THRESHOLDS`. No hardcodear valores numéricos en el código del notebook.

In [ ]:
rep = exceedance_report(df["caudal"], variable="caudal")
if rep.empty:
    print("Sin normas colombianas registradas para 'caudal'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["caudal"], method="linear")
print(f"Faltantes antes: {df["caudal"].isna().sum()} | "
      f"después: {df_clean["caudal"].isna().sum()}")

## 7. Modelos predictivos — NSE y KGE como métricas del POMCA

### ¿Por qué NSE y KGE en el contexto del POMCA?

El POMCA utiliza los pronósticos de caudal para dos propósitos distintos: (1) establecer el balance oferta-demanda en el horizonte de planificación, y (2) identificar los escenarios extremos (sequías e inundaciones) para la gestión del riesgo. Las métricas genéricas como RMSE no miden bien ninguno de estos dos objetivos.

### NSE (Nash-Sutcliffe Efficiency) — Estándar hidrológico colombiano

El NSE es el estándar de facto para la evaluación de modelos hidrológicos en Colombia. El IDEAM lo utiliza como criterio de aceptación en la calibración de los modelos SWAT y HEC-HMS usados en los POMCA:

| NSE | Evaluación IDEAM | Uso en POMCA |
|---|---|---|
| > 0.75 | Muy bueno | Modelo válido para proyecciones y balance hídrico |
| 0.65 – 0.75 | Satisfactorio | Aceptable para escenarios exploratorios |
| 0.50 – 0.65 | Aceptable con reservas | Solo para análisis preliminar |
| < 0.50 | Insatisfactorio | No publicar ni usar en el POMCA |

### KGE — Diagnóstico adicional

Para el POMCA, la descomposición del KGE permite un diagnóstico más preciso:
- Si **α < 0.7**: el modelo subestima la variabilidad — peligroso para escenarios de inundación (subestima los caudales máximos).
- Si **β < 0.9**: el modelo subestima el volumen medio — el balance oferta-demanda quedará sesgado hacia mayor disponibilidad aparente.
- Si **r < 0.8**: el modelo no captura la estacionalidad correctamente — las proyecciones mensuales del IUA serán poco confiables.

### ¿Qué modelo elegir para el POMCA?

- **SARIMA (baseline):** Captura estacionalidad y autocorrelación. Adecuado para proyecciones de caudal medio.
- **SARIMAX + ENSO:** Mejora significativamente en cuencas con alta señal ENSO. Recomendado para escenarios de cambio climático.
- **Prophet:** Robusto ante brechas en los datos. Útil para cuencas con series discontinuas.
- **XGBoost + características temporales:** Puede capturar no-linealidades como el efecto umbral del IUA.

> **ADR-003:** NSE y KGE son métricas primarias. RMSE puede calcularse como referencia, pero no debe ser el criterio de selección del modelo final para el POMCA.

In [ ]:
ts = df_clean.set_index("fecha")["caudal"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones metodológicas

### Síntesis del ciclo estadístico para el POMCA

Este notebook implementó el ciclo estadístico completo para la línea `pomca`:

1. **Validación de dominio** (`validate()`): verificación de rangos de caudal para cuencas hidrográficas.
2. **EDA**: caracterización del régimen hidrológico (bimodal, unimodal, equinoccial).
3. **ENSO con lag** (`enso_lagged()`): incorporación de la variabilidad climática interanual.
4. **Estacionariedad** (ADF + KPSS): verificación pre-ARIMA (ADR-004).
5. **Tendencia** (Mann-Kendall): detección de tendencias en el caudal para el diagnóstico del POMCA.
6. **Excedencias**: evaluación del IUA mensual contra umbrales de gestión del ENA.
7. **Modelos** (SARIMA, Prophet, XGBoost): proyecciones de caudal con métricas NSE, KGE.

### Decisiones metodológicas clave

| Decisión | Regla aplicada | ADR |
|---|---|---|
| No clipping de caudales extremos | Crecientes La Niña = señal real para gestión del riesgo | ADR-002 |
| ADF + KPSS obligatorios | Estacionariedad pre-ARIMA para proyecciones 10 años | ADR-004 |
| Umbrales IUA/IRH en config.py | No hardcodear umbrales del ENA en notebooks | ADR-005 |
| ENSO con lag por cuenca | Tiempo de respuesta hidrológica al ONI | ADR-007 |
| NSE y KGE primarios | Estándar IDEAM para evaluación de modelos hidrológicos | ADR-003 |

### Hallazgos de referencia (datos sintéticos)

Al trabajar con datos reales del POMCA, documentar:
- Régimen hidrológico identificado (bimodal, unimodal, equinoccial).
- Resultado ADF/KPSS y parámetros d/D del SARIMA seleccionado.
- Tendencia Mann-Kendall y proyección del caudal a 10 años.
- IUA mensual: ¿cuántos meses supera el 50%? ¿Alguno supera el 100%?
- Mejor modelo por NSE y KGE.
- Registrar decisiones en `docs/decisiones.md`.

### Normativa y referencias
- `docs/fuentes/pomca.md` — Ficha técnica completa del POMCA
- Decreto 1640 de 2012 / Decreto 1076 de 2015 — Marco legal POMCA
- IDEAM — Estudio Nacional del Agua (ENA) — IUA, IRH, IVH
- IDEAM — DHIME y SIRH — Fuentes de datos hidrológicos

## 9. Cómo adaptar a datos reales

Esta sección es la guía práctica para equipos técnicos de CARs o consultoras que estén trabajando en la elaboración o actualización de un POMCA.

### Paso 1 — Obtener los datos

**Fuentes recomendadas para el diagnóstico hídrico del POMCA:**

| Tipo de dato | Fuente | Cómo acceder |
|---|---|---|
| Caudales medios diarios | IDEAM — DHIME | dhime.ideam.gov.co (requiere cuenta institucional) |
| Demanda hídrica por cuenca | IDEAM — SIRH / ENA | sirh.ideam.gov.co |
| Precipitación | IDEAM — DHIME | Estaciones en el área de la cuenca |
| Coberturas (Corine Land Cover) | IGAC / SIAC | ide.dane.gov.co / siac.gov.co |
| IUA, IRH, IVH por subzona | IDEAM — Estudio Nacional del Agua | Documento ENA vigente (PDF + tablas) |

### Paso 2 — Preparar el archivo CSV de caudales

```
fecha,caudal,precipitacion,demanda,estacion
2000-01-01,45.2,80.5,8.5,21207070
2000-02-01,38.7,65.0,7.2,21207070
...
```

- `fecha`: formato ISO (`YYYY-MM-DD`), primer día del mes para series mensuales.
- `caudal`: caudal medio mensual (m³/s) — oferta hídrica superficial.
- `demanda`: demanda hídrica mensual (m³/s) — requerida para calcular IUA.
- `estacion`: código IDEAM de la estación hidrológica.
- Guardar en `data/raw/pomca.csv`.

### Paso 3 — Activar la carga real

```python
df = load_csv("data/raw/pomca.csv", date_col="fecha")
```

### Paso 4 — Calcular el IUA mensual

Una vez cargados los datos con columna `demanda`:

```python
from estadistica_ambiental.config import IUA_THRESHOLDS

# Oferta disponible = caudal total - caudal ambiental (25% del caudal medio)
caudal_ambiental = df["caudal"].mean() * 0.25
df["oferta_disponible"] = df["caudal"] - caudal_ambiental
df["iua"] = (df["demanda"] / df["oferta_disponible"]) * 100

# Categorización según config
df["categoria_iua"] = pd.cut(
    df["iua"],
    bins=[0, 20, 50, 100, float("inf")],
    labels=["Bajo", "Moderado", "Alto", "Crítico"]
)
print(df.groupby("categoria_iua").size())
```

### Paso 5 — Múltiples estaciones de la cuenca

El POMCA generalmente incluye varias estaciones hidrológicas a lo largo de la cuenca. Para analizarlas en conjunto:

```python
for estacion_id in df["estacion"].unique():
    ts = df[df["estacion"] == estacion_id].set_index("fecha")["caudal"]
    print(f"\n--- Estación {estacion_id} ---")
    stationarity_report(ts)
```

### Checklist de verificación antes de entregar al POMCA

- [ ] Mínimo 20 años de serie de caudal disponible.
- [ ] ADF + KPSS ejecutados (ADR-004).
- [ ] ENSO incorporado con `enso_lagged()` (ADR-007).
- [ ] IUA mensual calculado y categorizado con `config.IUA_THRESHOLDS` (ADR-005).
- [ ] Tendencia Mann-Kendall documentada.
- [ ] NSE > 0.65 en el modelo de proyección (ADR-003).
- [ ] Decisiones metodológicas registradas en `docs/decisiones.md`.